## EDA - Classes Distribution 

In [3]:
import os
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
data_path = "/mnt/raid5/neemias/PerceptSent-LLM-approach/data/minigpt4-classify"

# Get all CSV files in the directory
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]

# Function to format filename for display
def format_filename(filename):
    # Remove extension and prefix
    name = filename.replace('percept_dataset_', '').replace('.csv', '')
    
    # Split by underscore
    parts = name.split('_')
    
    # Extract alpha/sigma value and prompt part
    alpha_part = parts[0].replace('alpha', 'Sigma')
    prompt_part = parts[1].upper()
    
    # Handle prompt variations
    if 'neg' in prompt_part:
        prompt_suffix = 'Negative'
    elif 'plus' in prompt_part:
        prompt_suffix = 'Positive'
    else:
        prompt_suffix = ''
    
    prompt_num = prompt_part.replace('neg', '').replace('plus', '')
    
    return f"PerceptSent Dataset {alpha_part} {prompt_num} {prompt_suffix}".strip()

# Function to get label mapping based on prompt type
def get_label_mapping(filename):
    if 'p5' in filename.lower():
        return {4: "Positive", 3: "SlightlyPositive", 2: "Neutral", 1: "SlightlyNegative", 0: "Negative"}
    elif 'p2plus' in filename.lower():
        return {1: "Negative/SlightlyNegative", 0: "Positive/Neutral/SlightlyPositive"}
    elif 'p2neg' in filename.lower():
        return {1: "Positive/SlightlyPositive", 0: "Negative/Neutral/SlightlyNegative"}
    elif 'p3' in filename.lower():
        return {2: "Positive", 1: "Negative", 0: "Neutral"}
    else:
        return {}

# Function to get color based on sentiment label
def get_color(label):
    label_lower = label.lower()
    if 'positive' in label_lower and 'negative' not in label_lower:
        if 'slightly' in label_lower:
            return '#90EE90'  # Light green
        else:
            return '#228B22'  # Green
    elif 'negative' in label_lower and 'positive' not in label_lower:
        if 'slightly' in label_lower:
            return '#FFA07A'  # Light red
        else:
            return '#DC143C'  # Red
    elif 'neutral' in label_lower:
        return '#FFD700'  # Gold/Yellow
    else:
        # Mixed sentiment (contains both positive and negative)
        return '#808080'  # Gray
    
# Create subplots for each dataframe
fig, axes = plt.subplots(len(csv_files), 1, figsize=(10, 5 * len(csv_files)))

# Handle case when there's only one file
if len(csv_files) == 1:
    axes = [axes]

for idx, file in enumerate(csv_files):
    # Load dataframe
    df = pd.read_csv(os.path.join(data_path, file))
    
    # Get the class column
    class_col = df.columns[-1]
    
    # Get class counts and percentages
    class_counts = df[class_col].value_counts()
    class_percentages = (class_counts / len(df)) * 100
    
    # Get label mapping for this file
    label_mapping = get_label_mapping(file)
    
    # Map numeric labels to text labels
    text_labels = [label_mapping.get(label, str(label)) for label in class_counts.index]
    
    # Get colors for each bar
    colors = [get_color(label) for label in text_labels]
    
    # Create bar plot with colors
    bars = axes[idx].bar(range(len(class_counts)), class_counts.values, color=colors)
    axes[idx].set_xticks(range(len(class_counts)))
    axes[idx].set_xticklabels(text_labels, rotation=45, ha='right')
    
    axes[idx].set_ylabel('Count')
    axes[idx].set_title(f'Class Distribution - {format_filename(file)}')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add percentage labels on bars
    for i, (bar, count, pct) in enumerate(zip(bars, class_counts.values, class_percentages.values)):
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{pct:.1f}%\n({count})',
                      ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Distribution For Regression

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

# Path to where you saved the regression CSVs
data_path = "../data/gpt4-openai-regression/"

# Get all Regression CSV files
csv_files = [f for f in os.listdir(data_path) if f.startswith('percept_dataset_regression') and f.endswith('.csv')]
csv_files.sort() # Ensure consistent order

def format_filename(filename):
    """Parses filename to create a readable title."""
    name = filename.replace('percept_dataset_regression_', '').replace('.csv', '')
    return f"Regression Distribution: {name.upper()}"

def get_scale_max(filename):
    """Returns the maximum possible score based on the approach."""
    if 'p5' in filename: return 4.0
    if 'p3' in filename: return 2.0
    return 1.0 # Default for p2plus/p2neg

# Create subplots
fig, axes = plt.subplots(len(csv_files), 1, figsize=(12, 6 * len(csv_files)))
if len(csv_files) == 1: axes = [axes]

# Colormap: Red (Negative) -> Yellow (Neutral) -> Green (Positive)
cmap = cm.get_cmap('RdYlGn')

for idx, file in enumerate(csv_files):
    # Load Data
    file_path = os.path.join(data_path, file)
    df = pd.read_csv(file_path)
    
    # Identify the target column (could be 'sentiment_score' or 'sentiment')
    target_col = 'sentiment_score' if 'sentiment_score' in df.columns else 'sentiment'
    
    # Get unique values and their counts (sorted by score)
    # rounding to 2 decimals ensures floating point artifacts don't create duplicate bars
    val_counts = df[target_col].round(2).value_counts().sort_index()
    
    # Prepare Data for Plotting
    scores = val_counts.index.values
    counts = val_counts.values
    percentages = (counts / len(df)) * 100
    
    # Dynamic Coloring
    max_scale = get_scale_max(file)
    # Normalize scores 0 to 1 for the colormap
    norm_scores = scores / max_scale 
    bar_colors = [cmap(s) for s in norm_scores]

    # Plot
    bars = axes[idx].bar(range(len(scores)), counts, color=bar_colors, edgecolor='black', alpha=0.8)
    
    # X-Axis formatting
    axes[idx].set_xticks(range(len(scores)))
    axes[idx].set_xticklabels([f"{s:.2f}" for s in scores], rotation=0)
    
    # Labels
    axes[idx].set_ylabel('Count')
    axes[idx].set_xlabel('Regression Score')
    axes[idx].set_title(format_filename(file), fontsize=14, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3, linestyle='--')
    
    # Annotations (Count + Percentage)
    for bar, count, pct in zip(bars, counts, percentages):
        height = bar.get_height()
        axes[idx].text(
            bar.get_x() + bar.get_width()/2., 
            height + (max(counts)*0.01), # Slight offset
            f'{pct:.1f}%\n({count})',
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )

plt.tight_layout()
plt.show()